# 01 — Exploratory Data Analysis: HCP-Level Structure, Cohort Drift and Baseline Readiness

This notebook analyzes the curated silver layer from the perspective that:

- the prediction unit is the HCP
- each HCP contributes a fixed 86-week history
- labeled and unlabeled HCPs must be compared before any semi-supervised step
- a strong HCP-level tabular baseline should be built before moving into neural architectures


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp, skew
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plt.style.use("default")


## 1. Load silver assets


In [ ]:
DATA_DIR = Path("data")
SILVER_PATH = DATA_DIR / "silver_layer_longitudinal.parquet"
MANIFEST_PATH = DATA_DIR / "hcp_manifest.parquet"

if not SILVER_PATH.exists():
    raise FileNotFoundError(f"Silver dataset not found: {SILVER_PATH.resolve()}")
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"HCP manifest not found: {MANIFEST_PATH.resolve()}")

df = pd.read_parquet(SILVER_PATH)
hcp_manifest = pd.read_parquet(MANIFEST_PATH)

print("Silver shape:", df.shape)
print("Manifest shape:", hcp_manifest.shape)
print(df.head(2))


## 2. Structural overview


In [ ]:
overview = pd.DataFrame([
    {"metric": "row_count", "value": int(df.shape[0])},
    {"metric": "column_count", "value": int(df.shape[1])},
    {"metric": "unique_hcps", "value": int(df["NUEVO_ID"].nunique())},
    {"metric": "unique_weeks", "value": int(df["WEEK_ID"].nunique())},
    {"metric": "labeled_hcps", "value": int((hcp_manifest["IS_LABELED_HCP"] == 1).sum())},
    {"metric": "unlabeled_hcps", "value": int((hcp_manifest["IS_LABELED_HCP"] == 0).sum())},
    {"metric": "weeks_per_hcp_min", "value": int(df.groupby("NUEVO_ID").size().min())},
    {"metric": "weeks_per_hcp_median", "value": float(df.groupby("NUEVO_ID").size().median())},
    {"metric": "weeks_per_hcp_max", "value": int(df.groupby("NUEVO_ID").size().max())},
])
overview


In [ ]:
label_distribution = (
    hcp_manifest["ATSEG_HCP"]
    .value_counts(dropna=False)
    .rename_axis("ATSEG_HCP")
    .reset_index(name="hcp_count")
)
label_distribution["share"] = label_distribution["hcp_count"] / label_distribution["hcp_count"].sum()
label_distribution


In [ ]:
plot_df = label_distribution[label_distribution["ATSEG_HCP"].notna()].copy()

ax = plot_df.plot(
    kind="bar",
    x="ATSEG_HCP",
    y="hcp_count",
    figsize=(8, 5),
    legend=False,
    title="HCP-level labeled class distribution"
)
ax.set_xlabel("ATSEG_HCP")
ax.set_ylabel("Number of HCPs")
plt.tight_layout()
plt.show()


## 3. Structural integrity checks


In [ ]:
duplicate_pairs = int(df.duplicated(["NUEVO_ID", "WEEK_ID"]).sum())

rows_per_hcp = df.groupby("NUEVO_ID").size()
weeks_per_hcp = df.groupby("NUEVO_ID")["WEEK_ID"].nunique()
label_consistency = df.groupby("NUEVO_ID")["ATSEG_HCP"].nunique(dropna=True)

integrity = pd.DataFrame([
    {"check": "duplicate_hcp_week_pairs", "value": duplicate_pairs},
    {"check": "rows_per_hcp_min", "value": int(rows_per_hcp.min())},
    {"check": "rows_per_hcp_median", "value": float(rows_per_hcp.median())},
    {"check": "rows_per_hcp_max", "value": int(rows_per_hcp.max())},
    {"check": "weeks_per_hcp_min", "value": int(weeks_per_hcp.min())},
    {"check": "weeks_per_hcp_median", "value": float(weeks_per_hcp.median())},
    {"check": "weeks_per_hcp_max", "value": int(weeks_per_hcp.max())},
    {"check": "max_unique_labels_per_hcp", "value": int(label_consistency.max())},
])

integrity


In [ ]:
gap_distribution = (
    pd.to_datetime(df["WEEK_ID"])
    .groupby(df["NUEVO_ID"])
    .diff()
    .dropna()
    .dt.days
    .value_counts()
    .sort_index()
    .rename_axis("day_gap")
    .reset_index(name="count")
)

gap_distribution.head(10)


## 4. Feature groups for EDA


In [ ]:
specialty_dummy_cols = ["SPEC_GE", "SPEC_GPFM", "SPEC_IM", "SPEC_NRP", "SPEC_OTHER_SPEC", "SPEC_PHA"]
state_dummy_cols = ["STATE_1", "STATE_2", "STATE_3", "STATE_4", "STATE_5", "STATE_6", "STATE_7", "STATE_8", "STS_OTHER_STS"]
age_dummy_cols = ["(1940, 1960]", "(1960, 1980]", "(1980, 2000]", "(2000, 2020]", "(2020, 2030]"]
calendar_cols = ["YEAR", "QTR", "YEAR_QTR"]

trx_cols = [c for c in df.columns if c.endswith("_TRX")]
nrx_cols = [c for c in df.columns if c.endswith("_NRX")]
nbrx_cols = [c for c in df.columns if c.endswith("_NBRX")]
claim_cols = [c for c in df.columns if c.startswith("N_CLM")]
marketing_cols = ["RTE", "SAMPLES", "COPAY", "DIRECTMAIL", "SPK", "DETAILS"]
rolling_cols = [c for c in df.columns if "_R" in c and c.endswith("SUM")]
gidx_cols = [c for c in df.columns if c.endswith("_GIDX")]

dynamic_numeric_cols = [
    c for c in (trx_cols + nrx_cols + nbrx_cols + claim_cols + marketing_cols + rolling_cols + gidx_cols)
    if c in df.columns
]

static_dummy_cols = [c for c in (specialty_dummy_cols + state_dummy_cols + age_dummy_cols) if c in df.columns]

core_temporal_cols = [c for c in [
    "UC_TRX", "ORAL_TRX", "IL23_TRX",
    "UC_NRX", "ORAL_NRX", "IL23_NRX",
    "RTE", "SAMPLES", "COPAY", "DIRECTMAIL",
    "UC_TRX_R4_16SUM", "ORAL_NBRX_R4_29SUM", "IL23_NBRX_R4_29SUM"
] if c in df.columns]

print("Dynamic numeric columns:", len(dynamic_numeric_cols))
print("Static dummy columns:", len(static_dummy_cols))
print("Core temporal columns:", core_temporal_cols)


## 5. Missingness profile

The uploaded dataset has virtually no feature missingness.  
The only systematic missing field is the target, which is missing for unlabeled HCPs.


In [ ]:
missing_profile = (
    df.isna().mean()
    .sort_values(ascending=False)
    .rename_axis("column")
    .reset_index(name="missing_rate")
)

missing_profile[missing_profile["missing_rate"] > 0]


## 6. Build HCP-level aggregates for analysis

A professional baseline before neural networks should work at HCP level.  
This section builds a compact HCP table using weekly summaries.


In [ ]:
agg_mean = df.groupby("NUEVO_ID")[dynamic_numeric_cols].mean().add_suffix("__mean")
agg_std = df.groupby("NUEVO_ID")[dynamic_numeric_cols].std(ddof=0).fillna(0).add_suffix("__std")
agg_last = (
    df.sort_values(["NUEVO_ID", "WEEK_ID"])
    .groupby("NUEVO_ID")[dynamic_numeric_cols]
    .last()
    .add_suffix("__last")
)
agg_first = (
    df.sort_values(["NUEVO_ID", "WEEK_ID"])
    .groupby("NUEVO_ID")[dynamic_numeric_cols]
    .first()
    .add_suffix("__first")
)

hcp_features = pd.concat([agg_mean, agg_std, agg_last, agg_first], axis=1).reset_index()
hcp_features = hcp_features.merge(
    hcp_manifest[["NUEVO_ID", "ATSEG_HCP", "IS_LABELED_HCP", "SPECIALTY_GROUP", "STATE_GROUP", "AGE_RANGE_GROUP", "HCP_FOLD"]],
    on="NUEVO_ID",
    how="left"
)

for col in core_temporal_cols:
    hcp_features[f"{col}__delta_last_minus_first"] = hcp_features[f"{col}__last"] - hcp_features[f"{col}__first"]

print("HCP feature table shape:", hcp_features.shape)
hcp_features.head(3)


## 7. Labeled vs unlabeled cohort drift at HCP level

This section compares the labeled and unlabeled populations using HCP-level aggregated features.


In [ ]:
def compute_psi(expected, actual, bins=10):
    expected = pd.Series(expected).replace([np.inf, -np.inf], np.nan).dropna()
    actual = pd.Series(actual).replace([np.inf, -np.inf], np.nan).dropna()

    if expected.nunique() < 2 or actual.nunique() < 2:
        return np.nan

    quantiles = np.linspace(0, 1, bins + 1)
    breakpoints = np.unique(np.quantile(expected, quantiles))
    if len(breakpoints) < 3:
        return np.nan

    expected_bins = pd.cut(expected, bins=breakpoints, include_lowest=True)
    actual_bins = pd.cut(actual, bins=breakpoints, include_lowest=True)

    expected_dist = expected_bins.value_counts(normalize=True, sort=False).replace(0, 1e-6)
    actual_dist = actual_bins.value_counts(normalize=True, sort=False).replace(0, 1e-6)

    return float(((actual_dist - expected_dist) * np.log(actual_dist / expected_dist)).sum())

drift_feature_cols = [c for c in hcp_features.columns if c.endswith("__mean") or c.endswith("__std") or c.endswith("__last")]

labeled_hcp = hcp_features[hcp_features["IS_LABELED_HCP"] == 1]
unlabeled_hcp = hcp_features[hcp_features["IS_LABELED_HCP"] == 0]

drift_rows = []
for col in drift_feature_cols:
    x1 = labeled_hcp[col]
    x2 = unlabeled_hcp[col]
    ks_stat, ks_p = ks_2samp(x1, x2)
    psi_value = compute_psi(x1, x2, bins=10)

    drift_rows.append({
        "feature": col,
        "labeled_mean": float(x1.mean()),
        "unlabeled_mean": float(x2.mean()),
        "mean_diff": float(x2.mean() - x1.mean()),
        "ks_stat": float(ks_stat),
        "ks_pvalue": float(ks_p),
        "psi": psi_value,
    })

drift_df = pd.DataFrame(drift_rows).sort_values(["psi", "ks_stat"], ascending=[False, False])
drift_df.head(20)


In [ ]:
top_drift = drift_df.dropna(subset=["psi"]).head(15).sort_values("psi")
ax = top_drift.plot(kind="barh", x="feature", y="psi", figsize=(10, 8), legend=False, title="Top PSI values: labeled vs unlabeled HCPs")
ax.set_xlabel("PSI")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


## 8. Class-level descriptive statistics within labeled HCPs


In [ ]:
selected_summary_cols = [c for c in [
    "UC_TRX__mean", "ORAL_TRX__mean", "IL23_TRX__mean",
    "UC_NRX__mean", "ORAL_NRX__mean", "IL23_NRX__mean",
    "RTE__mean", "SAMPLES__mean", "COPAY__mean", "DIRECTMAIL__mean",
    "UC_TRX_R4_16SUM__last", "ORAL_NBRX_R4_29SUM__last", "IL23_NBRX_R4_29SUM__last"
] if c in hcp_features.columns]

class_summary = (
    labeled_hcp.groupby("ATSEG_HCP")[selected_summary_cols]
    .median()
    .T
)

class_summary


## 9. Temporal signal inspection

The goal here is not to train a sequence model yet, but to verify whether the 86-week histories carry meaningful temporal structure.


In [ ]:
if core_temporal_cols:
    temporal_class_profile = (
        df[df["IS_LABELED_HCP"] == 1]
        .groupby(["ATSEG_HCP", "WEEK_IDX"])[core_temporal_cols]
        .mean()
        .reset_index()
    )
    temporal_class_profile.head()
else:
    temporal_class_profile = pd.DataFrame()
    print("No core temporal columns were found.")


In [ ]:
if not temporal_class_profile.empty:
    feature_to_plot = core_temporal_cols[0]

    fig, ax = plt.subplots(figsize=(10, 5))
    for target_value, grp in temporal_class_profile.groupby("ATSEG_HCP"):
        ax.plot(grp["WEEK_IDX"], grp[feature_to_plot], label=target_value)

    ax.set_title(f"Average weekly trajectory by class: {feature_to_plot}")
    ax.set_xlabel("Week index within HCP history")
    ax.set_ylabel(feature_to_plot)
    ax.legend(title="ATSEG_HCP")
    plt.tight_layout()
    plt.show()


In [ ]:
lag_results = []
for col in core_temporal_cols:
    sampled = (
        df[df["IS_LABELED_HCP"] == 1]
        .groupby("NUEVO_ID")[col]
        .apply(lambda s: s.astype(float).autocorr(lag=1))
        .dropna()
    )
    lag_results.append({
        "feature": col,
        "mean_lag1_autocorr": float(sampled.mean()) if len(sampled) else np.nan,
        "median_lag1_autocorr": float(sampled.median()) if len(sampled) else np.nan,
    })

autocorr_df = pd.DataFrame(lag_results).sort_values("mean_lag1_autocorr", ascending=False)
autocorr_df


## 10. Distribution shape and outlier concentration at HCP level


In [ ]:
shape_rows = []
for col in [c for c in hcp_features.columns if c.endswith("__mean")]:
    series = hcp_features[col].replace([np.inf, -np.inf], np.nan).dropna()
    if len(series) == 0:
        continue

    p50 = series.quantile(0.50)
    p95 = series.quantile(0.95)
    p99 = series.quantile(0.99)

    shape_rows.append({
        "feature": col,
        "skewness": float(skew(series)),
        "p95": float(p95),
        "p99": float(p99),
        "p99_to_p50_ratio": float(p99 / p50) if p50 not in [0, np.nan] else np.nan,
    })

shape_df = pd.DataFrame(shape_rows).sort_values("skewness", ascending=False)
shape_df.head(20)


## 11. PCA overlap view for labeled HCPs

This is a compact visual diagnostic of class separability using HCP-level aggregated features.
It is not a modeling step.


In [ ]:
pca_input_cols = [c for c in hcp_features.columns if c.endswith("__mean") or c.endswith("__std") or c.endswith("__last")]
pca_df = labeled_hcp[["NUEVO_ID", "ATSEG_HCP"] + pca_input_cols].dropna()

X = pca_df[pca_input_cols].values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

pca_plot_df = pd.DataFrame({
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "ATSEG_HCP": pca_df["ATSEG_HCP"].values
})

fig, ax = plt.subplots(figsize=(9, 7))
for label, grp in pca_plot_df.groupby("ATSEG_HCP"):
    ax.scatter(grp["PC1"], grp["PC2"], s=16, alpha=0.6, label=label)

ax.set_title("PCA projection of HCP-level aggregated features")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)")
ax.legend(title="ATSEG_HCP")
plt.tight_layout()
plt.show()


## 12. EDA conclusions for the next notebook

The evidence in this notebook should be used to define the modeling roadmap:

1. the correct prediction unit is the HCP, not the individual week
2. class imbalance must be evaluated at HCP level
3. labeled and unlabeled HCPs should be compared before any pseudo-labeling
4. a strong tabular baseline should be built first using HCP-level aggregates
5. threshold tuning and calibration should be treated as first-class components of the baseline
6. neural sequence models should only be considered after the HCP-level baseline is stable and difficult to beat


In [ ]:
final_checks = {
    "n_hcp_features_rows": int(hcp_features.shape[0]),
    "n_hcp_features_cols": int(hcp_features.shape[1]),
    "n_labeled_hcp_features_rows": int(labeled_hcp.shape[0]),
    "n_unlabeled_hcp_features_rows": int(unlabeled_hcp.shape[0]),
    "top_5_drift_features": drift_df["feature"].head(5).tolist(),
    "top_5_skewed_hcp_features": shape_df["feature"].head(5).tolist(),
}

final_checks
